# ✉️ Questions écrites Schaerbeek — pipeline d'extraction

Même logique que le notebook PV (`PV_Schaerbeek_scrapper.ipynb`), en bien plus simple : chaque question écrite est un document court, mono-point, un seul appel Claude — pas de découpage en blocs ni d'audit de complétude (problèmes propres aux PV denses multi-points).

| Phase | Coût | Quand l'exécuter |
|---|---|---|
| 0 · Setup | gratuit | à chaque session |
| 1 · Pipeline (PDF non encore traités) | faible (1 appel Claude/PDF) | à chaque nouvel ajout de PDF dans `input/` |
| 2 · Copier le JSON dans le dépôt | gratuit | avant de committer |
| 3 · Commit → `main` | gratuit | quand la base est bonne |

**Dépose tes PDF de questions écrites dans `G:\Mon Drive\QE_Schaerbeek\input\`** avant d'exécuter la Phase 1 (dossier séparé de `PV_Schaerbeek`, créé s'il n'existe pas encore).

### 🔐 Secrets — AUCUNE clé en dur
Toutes les clés viennent du **gestionnaire de secrets Colab** (icône 🔑 à gauche) : `ANTHROPIC_API_KEY`, `GITHUB_TOKEN`.

## Phase 0 — Setup

In [ ]:
# 0.1 — Dépendances
!pip install -q pdfplumber anthropic tqdm

In [ ]:
# 0.2 — Drive + dépôt calé EXACTEMENT sur origin/main (aucune fusion, pas de divergence)
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.isdir('/content/pv-explorer-app'):
    !git clone https://github.com/pmeyssonnier/pv-explorer-app.git /content/pv-explorer-app
%cd /content/pv-explorer-app
!git fetch origin main
!git checkout -f -B main origin/main

# État de la base versionnée dans le dépôt (0 si pas encore de question publiée)
import json
import os as _os
_p = 'backend/questions_ecrites_schaerbeek.json'
if _os.path.exists(_p):
    d = json.load(open(_p))
    print(len(d['questions']), 'question(s) déjà publiée(s)')
else:
    print('Aucune question publiée pour le moment (fichier pas encore créé)')

In [ ]:
# 0.3 — 🔐 Secrets depuis Colab (jamais écrits dans le notebook)
# Helper TOLÉRANT AU NOM : essaie plusieurs noms possibles du secret, prend
# le 1er trouvé. Repli getpass (saisie masquée) hors-Colab ou si aucun trouvé.
import os

def get_secret(*names, prompt=None):
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None
    if userdata is not None:
        for n in names:
            try:
                return userdata.get(n)
            except Exception:
                continue   # nom absent / accès non accordé → essaie le suivant
    import getpass
    return getpass.getpass(prompt or f'{names[0]} : ')

os.environ['ANTHROPIC_API_KEY'] = get_secret(
    'ANTHROPIC_API_KEY', 'anthropic_api_key', 'ANTHROPIC_KEY',
    prompt='Clé Anthropic : ')

In [ ]:
# 0.4 — Imports pipeline + garde-fous
%cd /content/pv-explorer-app/pipeline

# ⚠️ Purge le cache d'imports : force la relecture des .py FRAÎCHEMENT checkout
import sys
for _m in ['questions_ecrites_extraction_pipeline', 'pv_extraction_pipeline']:
    sys.modules.pop(_m, None)

from questions_ecrites_extraction_pipeline import CONFIG, run_pipeline, load_database, get_pdf_list

print('MODEL      =', CONFIG['MODEL'])
print('INPUT_DIR  =', CONFIG['INPUT_DIR'])
pdfs = get_pdf_list(CONFIG['INPUT_DIR'])
print(len(pdfs), 'PDF trouvé(s) dans input/')

In [ ]:
# 0.5 — (optionnel, gratuit) vérifie que 2 PDF s'ouvrent, SANS appeler l'API
run_pipeline(max_files=2, dry_run=True)

## Phase 1 — Pipeline

`SKIP_ALREADY_DONE=True` (voir `CONFIG`) : les PDF déjà traités (suivis dans `progress.json` sur le Drive) sont ignorés automatiquement — relancer cette cellule après avoir ajouté de nouveaux PDF dans `input/` ne refacture que les nouveaux.

In [ ]:
stats = run_pipeline()
print(stats)

## Phase 2 — Copier le JSON dans le dépôt

In [ ]:
db = load_database()          # recharge la base réellement écrite sur le Drive
print(db['meta']['total_questions'], 'question(s) au total')

import shutil
SRC = CONFIG['DB_JSON_PATH']
DST = '/content/pv-explorer-app/backend/questions_ecrites_schaerbeek.json'
shutil.copy(SRC, DST)
print('copié →', DST)

## Phase 3 — Commit du JSON → `main`

Le token vient des **secrets Colab** (`GITHUB_TOKEN`). Il est injecté dans l'URL du remote *le temps du push*, puis **immédiatement retiré** du remote — jamais persistant, jamais écrit dans le notebook.

In [ ]:
%cd /content/pv-explorer-app

# Rester calé sur main sans perdre le JSON qu'on vient de copier
!git fetch origin main
!git stash -u    # met de côté le JSON modifié
!git checkout -f -B main origin/main
!git stash pop   # remet le JSON par-dessus la base à jour

In [ ]:
# get_secret (défini en Phase 0.3) essaie plusieurs noms : GITHUB_TOKEN,
# github_token, GITHUB_PAT… → aucun besoin de renommer ton secret.
TOKEN = get_secret('GITHUB_TOKEN', 'github_token', 'GITHUB_PAT', 'github_pat',
                   prompt='GitHub token : ')

!git config user.email 'pmeyssonnier@gmail.com'
!git config user.name  'pmeyssonnier'
!git add backend/questions_ecrites_schaerbeek.json
!git commit -m "data: intègre de nouvelles questions écrites (pipeline Colab)"

# push avec le token en clair UNIQUEMENT dans la commande, puis remote nettoyé
!git remote set-url origin https://{TOKEN}@github.com/pmeyssonnier/pv-explorer-app.git
!git push origin main
!git remote set-url origin https://github.com/pmeyssonnier/pv-explorer-app.git
del TOKEN
print('✅ JSON poussé sur main')

## Phase 4 — Thématiques des questions déjà publiées

`questions_ecrites_extraction_pipeline.py` extrait désormais une thématique par question, mais seulement pour les PDF traités APRÈS son ajout. Cette phase reclasse les questions déjà publiées (`thematiques` absent/vide) à partir du texte déjà en base — aucun besoin des PDF d'origine. **Idempotent** : relancer sans risque, ne retraite que les questions sans thème (sauf `--force`).

In [ ]:
# 4.1 — Recale le dépôt sur origin/main (récupère backfill_qe_thematiques.py
# et le SYSTEM_PROMPT à jour)
%cd /content/pv-explorer-app
!git fetch origin main
!git checkout -f -B main origin/main

In [ ]:
# 4.2 — Reclasse les questions sans thématique (même clé API que le reste du
# pipeline, voir Phase 0.3)
%cd /content/pv-explorer-app/backend
!python3 ../pipeline/backfill_qe_thematiques.py

# Pour tout reclasser, y compris les questions ayant déjà une thématique :
# !python3 ../pipeline/backfill_qe_thematiques.py --force

In [ ]:
# 4.3 — Commit + push (même mécanique de token éphémère qu'en Phase 3)
%cd /content/pv-explorer-app
TOKEN = get_secret('GITHUB_TOKEN', 'github_token', 'GITHUB_PAT', 'github_pat',
                   prompt='GitHub token : ')

!git config user.email 'pmeyssonnier@gmail.com'
!git config user.name  'pmeyssonnier'
!git add backend/questions_ecrites_schaerbeek.json
!git commit -m "data: classe les thematiques des questions écrites déjà publiées"

!git remote set-url origin https://{TOKEN}@github.com/pmeyssonnier/pv-explorer-app.git
!git push origin main
!git remote set-url origin https://github.com/pmeyssonnier/pv-explorer-app.git
del TOKEN
print('✅ JSON poussé sur main — relance ensuite les cellules 5.2/5.3 (Phase 5) pour réindexer Pinecone avec les nouvelles thématiques')

## Phase 5 — Indexation Pinecone (recherche via le chat)

Rend les questions écrites cherchables depuis l'onglet **Question** du site (`/ask`), en plus de l'onglet **Par élu·e** — voir `backend/index_qe.py`. Upsert **idempotent** par ID stable (`QE-{année}-{numéro}`) : relancer cette phase après un nouveau lot (Phases 1–3) est sans risque, aucun doublon.

Nécessite le secret Colab `PINECONE_API_KEY` (même clé que celle utilisée côté backend Render).

In [ ]:
# 5.1 — Dépendance + secret Pinecone
!pip install -q pinecone

import os
os.environ['PINECONE_API_KEY'] = get_secret(
    'PINECONE_API_KEY', 'pinecone_api_key', prompt='Clé Pinecone : ')

In [ ]:
# 5.2 — Recale le dépôt sur origin/main (récupère le JSON tout juste poussé en
# Phase 3, ou toute publication ponctuelle faite entre-temps via le panneau admin)
%cd /content/pv-explorer-app
!git fetch origin main
!git checkout -f -B main origin/main

In [ ]:
# 5.3 — Indexe (upsert idempotent — sans risque de doublon en relançant)
%cd /content/pv-explorer-app/backend
!python index_qe.py

# Pour ne réindexer QUE certaines années (ex. un nouveau lot ajouté pour
# 2022 seulement, sans retoucher aux années déjà indexées) :
# !python index_qe.py --only-year 2022